In [ ]:
import sys
from pathlib import Path
import pandas as pd
import matplotlib_inline
import tempfile
import subprocess

matplotlib_inline.backend_inline.set_matplotlib_formats("svg")
import re
import numpy as np
from tqdm.auto import tqdm

%matplotlib inline
pd.set_option("display.max_colwidth", None)
pd.set_option("display.max_rows", 500)
pd.set_option("display.max_columns", None)

sys.path.append(str(Path(snakemake.input["util"]).parent))
sys.path.append(str(Path(snakemake.input["display_util"]).parent))

In [ ]:
datanames = snakemake.params["datanames"]
data_paths = snakemake.input["raw_data"]
output = Path(snakemake.output[0])
output.mkdir(parents=True, exist_ok=True)
targetpop = pd.read_parquet(snakemake.input.targetpop)

## IDA Paper

In [ ]:
toignore = ["EBasisEmpfaengerNrNationalET"]
regexgroup = re.compile("(Empfaenger|Transplantation|Spender)")
instgroup = re.compile("(ET|DSO|IQTIG)$")
res = dict()

for dataname, data_path in tqdm(list(zip(datanames, data_paths))):
    curdata = pd.read_csv(data_path, sep=";", low_memory=False)
    nr_columns = [
        col
        for col in curdata.columns
        if ("Nr" in col or "Transplantationsnummer" in col) and col not in toignore
    ]
    curdata = curdata.loc[:, nr_columns]
    curdata = curdata.apply(lambda col: [list(col.dropna())])
    colnames = pd.Series(curdata.columns)
    group = colnames.str.extract(regexgroup, expand=False)
    assert not group.isna().any()
    filename = np.repeat(dataname, len(colnames))
    inst = colnames.str.extract(instgroup, expand=False)
    assert not inst.isna().any()
    columns = pd.MultiIndex.from_arrays(
        [filename, group, inst, colnames], names=("filename", "group", "inst", "col")
    )
    curdata.columns = columns
    res[dataname] = curdata
res = pd.concat(res.values(), axis=1).iloc[0, :]
assert (
    not res.index.to_frame()
    .reset_index(drop=True)
    .drop(columns="col")
    .duplicated()
    .any()
), "Duplicated insts"

In [ ]:
idcols_index = (
    res.index.to_frame()
    .reset_index(drop=True)
    .drop(columns="inst")
    .set_index("filename")
)
idcols_index["targetpop_col"] = idcols_index["group"].replace(
    {
        "Empfaenger": "recipient_et_id_et",
        "Spender": "donor_et_id_et",
        "Transplantation": "transplant_et_id",
    }
)
idcols_index.drop(columns="group", inplace=True)
idcols_index.head()

In [ ]:
def filter_read_data(dataname, data_path):
    curdata = pd.read_csv(data_path, sep=";", low_memory=False)
    relcols = idcols_index.loc[[dataname], :]
    filtercols = list()
    for index_r, row in relcols.iterrows():
        curcol = curdata.loc[:, row["col"]]
        allowed = targetpop.loc[:, row["targetpop_col"]]
        sel = curcol.isin(allowed) | curcol.isna()
        filtercols.append(sel)
    filtercols = pd.concat(filtercols, axis=1).all(axis=1)
    return (
        curdata.loc[filtercols, :]
        .dropna(axis=1, how="all")
        .dropna(axis=0, how="all")
        .copy()
    )

In [ ]:
res = dict()

for dataname, data_path in tqdm(list(zip(datanames, data_paths))):
    curdata = filter_read_data(dataname, data_path)
    nr_columns = [
        col
        for col in curdata.columns
        if ("Nr" in col or "Transplantationsnummer" in col) and col not in toignore
    ]
    curdata = curdata.loc[:, nr_columns]
    curdata = curdata.apply(lambda col: [list(col.dropna())])
    colnames = pd.Series(curdata.columns)
    group = colnames.str.extract(regexgroup, expand=False)
    assert not group.isna().any()
    filename = np.repeat(dataname, len(colnames))
    inst = colnames.str.extract(instgroup, expand=False)
    assert not inst.isna().any()
    columns = pd.MultiIndex.from_arrays(
        [filename, group, inst, colnames], names=("filename", "group", "inst", "col")
    )
    curdata.columns = columns
    res[dataname] = curdata
res = pd.concat(res.values(), axis=1).iloc[0, :]
assert (
    not res.index.to_frame()
    .reset_index(drop=True)
    .drop(columns="col")
    .duplicated()
    .any()
), "Duplicated insts"

In [ ]:
groupall = (
    res.groupby("group")
    .apply(lambda cols: len({entry for col in cols for entry in col}))
    .rename("Unique Entries")
    .to_frame()
)
display(groupall)
groupall.to_csv(output / "groupall.csv")

Frage: Wie tragen die Institute zu den Daten bei?

Antwort: Beschreibung der Verknüpfung der Tabellen mit den Instituten.

Frage: Welche Daten enthält das Register?

Antwort: Auflistung der Tabellen, die Teil des Exports sind und wie viele Patienten in den Tabellen enthalten sind.

In [ ]:
def summarize_group(group):
    numberofentries = group.apply(lambda col: len(set(col)))
    allentries = [entry for entries in group for entry in entries]
    groupeval = []
    for k in group.keys():
        allother = group.copy().drop(k)
        allother = {entry for entries in allother for entry in entries}
        remainder = set(allentries) - allother
        thisgroup = pd.Series(group.loc[k])
        thisgroup = thisgroup.value_counts()
        withrepeats = thisgroup[thisgroup > 1].sum()
        averageappearance = thisgroup.mean()
        uniquevals = len(remainder)
        mostfrequentrep = max(thisgroup)
        groupeval.append([uniquevals, withrepeats, averageappearance, mostfrequentrep])
    res = pd.DataFrame(groupeval)
    res.columns = [
        "Only from this Source",
        "With Repeats",
        "Average Appearance",
        "Most Frequent Repeat",
    ]
    res["group_members"] = len(set(allentries))
    res["repeating"] = (pd.Series(allentries).value_counts() > 1).sum()
    res["Number of Unique Instances"] = numberofentries.to_list()
    res.index = numberofentries.index
    res.set_index(["group_members", "repeating"], append=True, inplace=True)
    res.index = res.index.reorder_levels(
        ["filename", "group", "group_members", "repeating", "inst", "col"]
    )
    res = res.droplevel([0, 1], axis=0)
    return res


tosave = res.groupby(level=["filename", "group"]).apply(summarize_group)
display(tosave)
tosave.index.set_names(
    [
        "Table",
        "Content",
        "Number of Unique Instances in File",
        "Occuring more than one",
        "Data Source",
        "Column",
    ],
    inplace=True,
)
tosave.to_csv(output / "unique_entries.csv")

Frage: Wie tragen die Institute zu den Daten bei?
Antwort: Beschreibung der Verknüpfung der Tabellen mit den Instituten.

In [ ]:
def summarize_group(group: pd.Series):
    group.index = group.index.droplevel(["group"])
    allmembers = pd.Series(
        [m for m in {member for members in group for member in members}]
    ).drop_duplicates()
    df = group.apply(lambda x: allmembers.isin(x))
    df.index = df.index.reorder_levels(["filename", "inst", "col"])
    return df


tosave = res.groupby("group").apply(summarize_group)
for groupname, table in tosave.groupby(level="group"):
    table = table.transpose().dropna(how="all")
    table.columns = table.columns.droplevel("group")
    display(table.tail())
    table.to_csv(output / f"overlap_{groupname}.csv")

Welche Datentypen gibt es?

In [ ]:
unique_counts = list()


for dataname, data_path in tqdm(list(zip(datanames, data_paths))):
    curdata = filter_read_data(dataname, data_path)
    colnames = pd.Series(curdata.columns)

    nunique = curdata.nunique()
    dtypes = curdata.dtypes
    dtypes.name = "Data Type"
    nunique.name = "Unique Values (Ignoring NA)"
    val_counts = curdata.apply(
        lambda col: col.value_counts(normalize=True).head(3).reset_index(drop=True)
    )
    val_counts = val_counts.set_axis(
        [f"Freq of {i+1}th common" for i in range(val_counts.shape[0])], axis=0
    ).transpose()
    cur_res = pd.concat([nunique, dtypes, val_counts], axis=1)
    print(cur_res)

    filename = np.repeat(dataname, len(colnames))
    inst = colnames.str.extract(instgroup, expand=False)
    assert inst.notna().all()
    columns = pd.MultiIndex.from_arrays(
        [filename, inst, colnames], names=("filename", "inst", "col")
    )
    cur_res = cur_res.set_axis(columns, axis=0)
    unique_counts.append(cur_res)

unique_counts = pd.concat(unique_counts, axis=0)
unique_counts.to_csv(output / "unique_counts.csv")
display(unique_counts)

Frage: Wie viele Daten gibt es in dem longitudinalen Tabelle?
Antwort: Das variert je nach Tabelle

In [ ]:
def analyze_time(winsorize=False):
    alltimecols = dict()
    suffixes = ""
    for dataname, data_path in tqdm(list(zip(datanames, data_paths))):
        curdata = filter_read_data(dataname, data_path)
        datcols = [col for col in curdata.columns if "dat" in col.lower()]
        datcols = curdata.loc[:, datcols].dropna(how="all")
        datcols /= 365.25
        if winsorize:
            datcols = datcols.clip(
                lower=datcols.quantile(0.05), upper=datcols.quantile(0.95), axis=1
            )
        datcols = datcols.apply([pd.Series.min, pd.Series.max, pd.Series.std])
        datcols = datcols.loc[:, datcols.loc["min", :] != datcols.loc["max", :]].dropna(
            axis=1
        )
        if datcols.shape[0] > 0:
            colnames = pd.Series(datcols.columns)
            filename = np.repeat(dataname, len(colnames))
            inst = colnames.str.extract(instgroup, expand=False)
            columns = pd.MultiIndex.from_arrays(
                [filename, inst, colnames], names=("filename", "inst", "col")
            )
            datcols.columns = columns
            alltimecols[dataname] = datcols

    alltimecols = pd.concat(alltimecols.values(), axis=1).transpose()

    if winsorize:
        suffixes = ", 5% Winsorized)"
    alltimecols[f"Time Range (years{suffixes})"] = (
        alltimecols["max"] - alltimecols["min"]
    )
    if winsorize:
        suffixes = " (5% Winsorized)"
    alltimecols = alltimecols.rename(columns={"std": f"Standard Deviation {suffixes}"})
    alltimecols = alltimecols.drop(columns=["min", "max"])
    return alltimecols


alltimecols = pd.concat([analyze_time(), analyze_time(winsorize=True)], axis=1)
alltimecols.to_csv(output / "time_analysis.csv")
display(alltimecols)

Frage: Wie viele fehlende Daten gibt es?
Antwort: Im Register über 50% als fehlende Daten markiert. 

In [ ]:
missingdata = list()


def percentile(n):
    def percentile_(x):
        return x.quantile(n)

    percentile_.__name__ = "percentile_{:02.0f}".format(n * 100)
    return percentile_


for dataname, data_path in tqdm(list(zip(datanames, data_paths))):
    curdata = filter_read_data(dataname, data_path)
    colnames = pd.Series(curdata.columns)
    curdata = curdata.isna().sum() / curdata.shape[0]
    filename = np.repeat(dataname, len(colnames))
    inst = colnames.str.extract(instgroup, expand=False)
    assert inst.notna().all()
    columns = pd.MultiIndex.from_arrays(
        [filename, inst, colnames], names=("filename", "inst", "col")
    )
    curdata = curdata.set_axis(columns, axis=0)
    curdata.name = "Missing"
    missingdata.append(curdata)

missingdata = pd.concat(missingdata, axis=0).to_frame()
display(missingdata)

In [ ]:
missingdata_agg = missingdata.groupby(["filename", "inst"]).agg(
    [
        "median",
        percentile(0.25),
        percentile(0.75),
        lambda x: (x == 1).sum(),
        lambda x: (x == 0).sum(),
        "count",
    ]
)
missingdata_agg = missingdata_agg.droplevel(0, axis=1)
cols = missingdata_agg.columns.to_list()
cols[3:5] = ["All Missing", "All Present"]
missingdata_agg = missingdata_agg.set_axis(cols, axis=1)
missingdata_agg

Influx/Outflux:

In [ ]:
def flux_analysis(df):
    df = df.transpose()
    result_rr = pd.DataFrame(index=df.columns, columns=df.columns)
    result_rm = pd.DataFrame(index=df.columns, columns=df.columns)
    result_mr = pd.DataFrame(index=df.columns, columns=df.columns)
    result_mm = pd.DataFrame(index=df.columns, columns=df.columns)
    df = df.isna()

    # Iterate over each pair of columns
    for col1 in df.columns:
        for col2 in df.columns:
            col1_data = ~df[col1]
            col2_data = ~df[col2]
            result_rr.loc[col1, col2] = (col1_data & col2_data).sum()
            result_rm.loc[col1, col2] = (col1_data & ~col2_data).sum()
            result_mr.loc[col1, col2] = (~col1_data & col2_data).sum()
            result_mm.loc[col1, col2] = (~col1_data & ~col2_data).sum()

    # How much of the first column is missing when the second is missing (inbound), average
    ainb = (result_mr / (result_mr + result_mm)).sum(axis=1) / (df.shape[0] - 1)
    # How much of the first column is missing when the second is not missing (outbound), average
    aout = (result_rm / (result_rm + result_rr)).sum(axis=1) / (df.shape[0] - 1)
    # Number of pairs, where the first is present and the second is missing, divided by the number of pairs (when same percentage is missing, the variable might be more useful to impute)
    outflux = result_rm.sum(axis=1) / (result_rm + result_mm).sum(axis=1)
    # Number of pairs, where the first is missing and the second is present, divided by the number of pairs (when same percentage is missing, the variable might be easier to impute)
    influx = result_mr.sum(axis=1) / (result_mr + result_rr).sum(axis=1)
    missing = df.sum() / df.shape[0]
    res = pd.DataFrame(
        {
            "Inbound": ainb,
            "Outbound": aout,
            "Outflux": outflux,
            "Influx": influx,
            "Missing": missing,
        }
    )
    return res


missingdata = []

for dataname, data_path in tqdm(list(zip(datanames, data_paths))):
    curdata = filter_read_data(dataname, data_path)
    colnames = pd.Series(curdata.columns)
    filename = np.repeat(dataname, len(colnames))
    inst = colnames.str.extract(instgroup, expand=False)
    assert inst.notna().all()
    columns = pd.MultiIndex.from_arrays(
        [filename, inst, colnames], names=("filename", "inst", "col")
    )
    curdata = curdata.set_axis(columns, axis=1)
    missingdata.append(
        curdata.transpose().groupby(["filename", "inst"]).apply(flux_analysis)
    )

In [ ]:
missingdata = pd.concat(missingdata, axis=0).droplevel([0, 1], axis=0)
display(missingdata)
missingdata.to_csv(output / "missing_flux.csv")

Gibt es Inkonsistenzen zwischen den Instituten?

In [ ]:
from scipy.stats.contingency import association

red = []
upsetdata = []
all_comp_res = []

for dataname, data_path in tqdm(list(zip(datanames, data_paths))):
    curdata = filter_read_data(dataname, data_path)
    colnames = pd.Series(curdata.columns)
    filename = np.repeat(dataname, len(colnames))
    inst = colnames.str.extract(instgroup, expand=False)
    assert inst.notna().all()
    columns = pd.MultiIndex.from_arrays([filename, inst], names=("filename", "inst"))
    colnames.index = columns
    colnames = colnames.str.replace(instgroup, "", regex=True)
    colnames.name = "basename"
    colnames = colnames.reset_index().pivot(
        columns="inst", index=["filename", "basename"], values="inst"
    )
    recurrencecount = (~colnames.isna()).sum(axis=1)

    # We only need to do comparisons, if there are multiple institutions
    table_comp_res = []
    if colnames.shape[1] > 1:
        relevantcols = colnames[recurrencecount > 1]
        for (tablename, colbasename), row in relevantcols.iterrows():
            insts_to_compare = row.dropna().index.to_list()
            for i in range(len(insts_to_compare)):
                for j in range(i + 1, len(insts_to_compare)):
                    comp_res = dict()
                    comp_res["Table"] = tablename
                    comp_res["Column"] = colbasename
                    inst1 = insts_to_compare[i]
                    inst2 = insts_to_compare[j]
                    comp_res["Inst1"] = inst1
                    comp_res["Inst2"] = inst2
                    vals1 = curdata.loc[:, colbasename + inst1]
                    vals2 = curdata.loc[:, colbasename + inst2]
                    vals1_numeric = pd.api.types.is_numeric_dtype(vals1)
                    vals2_numeric = pd.api.types.is_numeric_dtype(vals2)
                    usable_sel = (~vals1.isna()) & (~vals2.isna())
                    comp_res["comparable"] = usable_sel.sum()
                    comp_res["comparable_rel"] = comp_res["comparable"] / vals1.size
                    comp_res["missing_1"] = vals1.isna().sum() / vals1.size
                    comp_res["missing_2"] = vals2.isna().sum() / vals2.size
                    comp_res["useprob_1"] = (
                        vals1.isna() & ~vals2.isna()
                    ).sum() / vals1.isna().sum()
                    comp_res["useprob_2"] = (
                        vals2.isna() & ~vals1.isna()
                    ).sum() / vals2.isna().sum()
                    equal = vals1.loc[usable_sel] == vals2.loc[usable_sel]
                    if equal.size > 0:
                        comp_res["accuracy"] = equal.sum() / equal.size
                    else:
                        comp_res["accuracy"] = np.nan
                    comp_res["mismatch_type"] = False
                    if vals1_numeric != vals2_numeric:
                        comp_res["mismatch_type"] = True
                    elif vals1_numeric:
                        stds = (
                            pd.concat([vals1, vals2], axis=1).loc[usable_sel, :].std()
                        )
                        if (comp_res["comparable"] > 1) and (stds > 0).all():
                            comp_res["pearson_r"] = vals1.corr(vals2, method="pearson")
                        else:
                            comp_res["pearson_r"] = np.nan
                    else:
                        contingency_table = pd.crosstab(vals1, vals2)
                        # If there is only one row or column, the Cramer's V is not defined, but
                        if (contingency_table.shape[0] == 1) and comp_res[
                            "accuracy"
                        ] == 1:
                            comp_res["crammers_v"] = np.inf
                        elif contingency_table.shape[0] > 1:
                            comp_res["crammers_v"] = association(
                                contingency_table, method="cramer"
                            )
                        else:
                            comp_res["crammers_v"] = np.nan
                    table_comp_res.append(comp_res)
    if table_comp_res:
        table_comp_res = pd.DataFrame(table_comp_res)
        table_comp_res["Table"] = dataname
        all_comp_res.append(table_comp_res)
    instcounts = (~colnames.isna()).sum(axis=0)
    upsetdata.append(~colnames.isna())
    res = pd.DataFrame(
        {
            "Total Columns": colnames.size,
            **{f"{k} Columns": v for k, v in instcounts.items()},
            "Redundant Groups": (recurrencecount > 1).sum(),
            "Columns in Redundant Groups": recurrencecount[recurrencecount > 1].sum(),
            "Unique Columns": recurrencecount[recurrencecount == 1].sum(),
        },
        index=[dataname],
    )
    red.append(res)
red = pd.concat(red).fillna(0).astype("int")
display(red)
red.index.name = "Table"
red.to_csv(output / "redundant_columns.csv")

In [ ]:
pd.concat(all_comp_res).to_csv(output / "column_comparison.csv", index=False)

In [ ]:
up = pd.concat(upsetdata).fillna(False)
display(up)
up.to_csv(output / "redundant_columns_up.csv")

Methode aus https://bmjopen.bmj.com/content/bmjopen/5/6/e007450.full.pdf

In [ ]:
def missing_perc_method(toprocess, pdfloc):
    with tempfile.NamedTemporaryFile() as scriptfile, tempfile.NamedTemporaryFile() as datafile, tempfile.NamedTemporaryFile() as outfile:
        toprocess.to_csv(datafile.name, header=True, index=False, na_rep="NA")

        scriptfile.write(b"""
            library(rpart)
            library(dplyr)
            library(rpart.plot)
            args <- commandArgs(trailingOnly = TRUE)
            data <- read.csv(args[1], na.strings = c("", "NA"))
            print(data)

            data <- data %>% mutate_if(is.character, as.factor)
            outfile <- args[2]
            pdffile <- args[3]

            tosplit <- data
            tosplit$target <- rowSums(is.na(tosplit))/ncol(tosplit)
            set.seed(123)
            train <- sample(1:nrow(tosplit), 0.7*nrow(tosplit))
            test <- setdiff(1:nrow(tosplit), train)
            train <- tosplit[train, ]
            test <- tosplit[test,]
            if(min(train$target) == max(train$target)) {
                df <- data.frame(train_rmse = NA, test_rmse = NA, imps = NA)
                print(df)
                write.csv(df, outfile, row.names = FALSE)
            }else{
                rpart_fit <- rpart(target ~ ., data = train, method = "anova")
                testrmse <- sqrt(mean((predict(rpart_fit, test) - test$target)^2))
                trainrmse <- sqrt(mean((predict(rpart_fit, train) - train$target)^2))

                imps <- summary(rpart_fit)$variable.importance
                imps_json <- paste0("{", paste0(sprintf('"%s": %s', names(imps), imps), collapse = ", "), "}")
                df <- data.frame(train_rmse = trainrmse, test_rmse = testrmse, imps = imps_json)
                print(df)
                write.csv(df, outfile, row.names = FALSE)

                pdf(pdffile)
                splitfun <- function(x, labs, digits, varlen, faclen) {
                    labs <- gsub(",", " ", labs)
                    for(i in 1:length(labs)) {
                    # split labs[i] into multiple lines
                    labs[i] <- paste(strwrap(labs[i], width = 32), collapse = "\n")
                    }
                    labs
                }
                rpart.plot(rpart_fit, type = 4, clip.right.labs = FALSE, branch = .3, under = TRUE, tweak=0.5, split.fun=splitfun)
                dev.off()
            }
        """)
        scriptfile.flush()
        subprocess.run(
            ["Rscript", scriptfile.name, datafile.name, outfile.name, pdfloc],
            capture_output=True,
            text=True,
        )
        outfile.seek(0)
        res = pd.read_csv(outfile)
        return res


allres = []

for dataname, data_path in tqdm(list(zip(datanames, data_paths))):
    curdata = filter_read_data(dataname, data_path)
    colnames = pd.Series(curdata.columns)
    filename = np.repeat(dataname, len(colnames))
    inst = colnames.str.extract(instgroup, expand=False)
    assert inst.notna().all()
    columns = pd.MultiIndex.from_arrays(
        [filename, inst, colnames], names=("filename", "inst", "col")
    )
    curdata = curdata.set_axis(columns, axis=1)
    for group, toprocess in curdata.transpose().groupby("inst"):
        toprocess = toprocess.transpose()
        pdfloc = output / "missplots" / f"{dataname}_{group}.pdf"
        pdfloc.parent.mkdir(parents=True, exist_ok=True)
        toprocess = toprocess.droplevel([0, 1], axis=1)
        nunique = toprocess.nunique()
        toprocess = toprocess.loc[
            :, (nunique > 1) & ((toprocess.dtypes != "O") | (nunique <= 250))
        ]
        toprocess = toprocess.dropna(how="all", axis=0).dropna(how="all", axis=1)
        if toprocess.shape[0] == 0:
            res = pd.DataFrame(
                {"train_rmse": np.nan, "test_rmse": np.nan, "imps": np.nan}, index=[0]
            )
        else:
            res = missing_perc_method(toprocess, pdfloc)
        res = res.set_axis(
            pd.MultiIndex.from_tuples([(dataname, group)], names=["filename", "inst"]),
            axis=0,
        )
        allres.append(res)
allres = pd.concat(allres)
display(allres)
allres.to_csv(output / "missing_frac_pred_method_res.csv")

### Processed Data

In [ ]:
userfriendly = pd.read_parquet(snakemake.input["userfriendly"]).reset_index(drop=True)
userfriendly.columns.to_list()

In [ ]:
userfriendly_missing = (
    userfriendly.isna().droplevel(-1, axis=1).sum(axis=0).rename("missing").to_frame()
)
userfriendly_missing.to_csv(output / "userfriendly_missing.csv", index=True)

In [ ]:
userfriendly_missing_ds = (
    userfriendly.isna()
    .droplevel(-1, axis=1)
    .transpose()
    .groupby("dataset")
    .any()
    .transpose()
)
userfriendly_missing_ds.to_csv(output / "userfriendly_missing_ds.csv", index=False)

In [ ]:
(
    userfriendly.loc[
        :,
        ("organ_entnahme_niere", "cold_ischemia_time_min", "feature"),
    ]
    / 60
).describe()

In [ ]:
new_nat = {
    ("empfaenger", "birthdate", "feature"): "recipient_age",
    ("empfaenger", "sex", "feature"): "recipient_sex",
    ("empfaenger", "origin_country", "feature"): "recipient_origin_country",
    (
        "warteliste_niere",
        "kidney_base_disease_text__timeseries_keep",
        "feature",
    ): "recipient_disease",
    (
        "empfaenger_dringlichkeit",
        "date__timeseries_range",
        "feature",
    ): "recipient_wait_time",
    ("spender_postmortem", "age", "feature"): "donor_age_1",
    ("spender_postmortem", "birthdate", "feature"): "donor_age_2",
    ("spender_postmortem", "sex", "feature"): "donor_sex",
    (
        "spender_postmortem_diagnosen",
        "death_reason_icd_code_dso",
        "feature",
    ): "donor_death_reason",
    ("transplantation", "assignment_program", "feature"): "assignment_program",
    (
        "organ_entnahme_niere",
        "cold_ischemia_time_min",
        "feature",
    ): "cold_ischemia_time_min",
}
# TODO eventuell doch ischämie aus transpl. nehmen
toanalyse = userfriendly.loc[:, new_nat.keys()].set_axis(new_nat.values(), axis=1)
toanalyse["recipient_age"] = toanalyse["recipient_age"] / -365.25
toanalyse["donor_age_2"] = toanalyse["donor_age_2"] / -365.25
toanalyse["donor_age"] = toanalyse.loc[:, ["donor_age_1", "donor_age_2"]].mean(
    axis=1, skipna=True
)
toanalyse = toanalyse.drop(columns=["donor_age_1", "donor_age_2"])
tofix = ~toanalyse["recipient_disease"].isna() & toanalyse[
    "recipient_disease"
].str.contains(";;")
toanalyse.loc[tofix, "recipient_disease"] = (
    toanalyse.loc[tofix, "recipient_disease"].str.split(";;").str[0]
)
toanalyse["recipient_disease"] = toanalyse["recipient_disease"].replace({"nan": np.nan})
toanalyse["recipient_wait_time"] = toanalyse["recipient_wait_time"] / 365.25
toanalyse = toanalyse.sort_index(axis=1)
toanalyse.to_csv(output / "userfriendly.csv", index=False)
display(toanalyse.head(3))

In [ ]:
toanalyse["recipient_disease"].value_counts().to_frame()

In [ ]:
datesrc = userfriendly.droplevel(-1, axis=1).loc[
    :,
    [
        ("empfaenger", "death_date"),
        ("followup_niere", "date__timeseries_keep"),
        ("followup_niere", "death_date__timeseries_keep"),
        ("transplantation", "postop_organ_failure_date"),
        ("followup_niere", "graft_failure_date__timeseries_keep"),
        ("followup_niere", "patient_died__timeseries_keep"),
        ("transplantation", "last_followup_date"),
        ("followup_niere", "graft_failure__timeseries_keep"),
    ],
]
datesrc = datesrc.set_axis(["::".join(coln) for coln in datesrc.columns], axis=1)
display(datesrc.head())
datesrc.to_csv(output / "dates.csv", index=False)

## Other

In [ ]:
consolidated = pd.read_parquet(snakemake.input["consolidated"])

In [ ]:
filtered_columns = consolidated.filter(like="program", axis=1)
filtered_columns

In [ ]:
import seaborn as sns

ageplotdata = (
    consolidated.reset_index(drop=True)
    .loc[
        :,
        [
            ("spender_postmortem", "age"),
            ("empfaenger", "birthdate"),
            ("transplantation", "assignment_program"),
        ],
    ]
    .copy()
)
ageplotdata.columns = ageplotdata.columns.droplevel(level=1)
ageplotdata["empfaenger"] /= -365.25
ageplotdata.rename(
    columns={
        "spender_postmortem": "donor_age",
        "empfaenger": "recipient_age",
        "transplantation": "program",
    },
    inplace=True,
)
sns.scatterplot(data=ageplotdata, x="donor_age", y="recipient_age", hue="program")

In [ ]:
data = pd.Series(list(snakemake.input.data))
names = pd.Series(snakemake.params.datanames)


empimm = pd.read_parquet(data[names == "empfaenger_immunologie"].iloc[0])
empimm = pd.merge(
    empimm,
    targetpop,
    left_index=True,
    right_on=["recipient_et_id_et"],
    how="inner",
    validate="m:1",
).set_index(list(targetpop.columns))

empimm["transplant_date"] = empimm.index.get_level_values("recipient_op_date")

hla = (
    empimm[empimm["result_type"] == "HLA Typing"]
    .dropna(how="all", axis=1)
    .drop(columns="result_type")
)
hla = hla.groupby(empimm.index.names, dropna=False).tail(1)

In [ ]:
hla["missing"] = hla["hla_phenotyping"].isna()
toplot = (
    hla[["enter_date", "transplant_date", "missing"]]
    .reset_index(drop=True)
    .sort_values("transplant_date")
)
toplot["transplant_date"] = pd.to_datetime(
    toplot["transplant_date"], origin="unix", unit="D"
)
toplot.set_index("transplant_date", inplace=True)

In [ ]:
from datetime import timedelta

import matplotlib.pyplot as plt

window_width = timedelta(days=365.25 / 12)  # Specify the window width as a timedelta

# Calculate the rolling mean
rolling_mean = toplot["missing"].rolling(window=window_width).mean()

# Plot the rolling mean
plt.plot(toplot.index, rolling_mean)
plt.xlabel("Enter Date")
plt.ylabel("Mean of Missing")
plt.title("Sliding Window Plot of Mean of Missing over Enter Date")
plt.show()